# MultiThreading

https://docs.python.org/3/library/threading.html

## 1. Qu’est-ce qu’un thread ?

Un thread est une unité d’exécution au sein d’un programme.
Un programme peut contenir plusieurs threads, qui s’exécutent en "pseudo-parallèle".

En Python, ce parallélisme est géré par le GIL (Global Interpreter Lock), ce qui signifie que :

- Plusieurs threads peuvent exister en meme temps, mais un seul peut exectuer du bytecode Python à la fois
- en revanche, les threads peuvent s’exécuter en parallèle pendant des opérations `I/O` (réseau, disque, sleep…), car ces opérations libèrent le GIL



Exemples de bonnes utilisations des threads :

- télécharger plusieurs fichiers
- gérer plusieurs connexions réseau
- exécuter une tâche en arrière-plan
- surveiller un événement

les threads sont très utiles pour les tâches I/O : réseau, fichiers, entrées-sorties, attente

## 2. Créer et démarrer un Thread

Imaginons disposer d'une fonction `worker` qui effectue des calculs chaque seconde.

In [1]:
import time

def worker():
    for i in range(10):
        print(i)
        time.sleep(1)
        

In [2]:
worker()

0
1
2
3
4
5
6
7
8
9


Maintenant, si nous voulons effectuer d'autres calculs entre chaque temps d'attente, nous pouvons le faire en placant cette fonction dans un `thread`

Nous allons donc utiliser le module `threading`

1. On créer un objet issu de la classe `Thread` et on l'assigne a notre fonction
2. `.start()` → démarre le thread (la fonction s'exécute dans un thread séparé)

In [3]:
import threading

In [ ]:
t = threading.Thread(target=worker)
t.start()

0


1
2
3
4
5
6
7
8
9


Une fois le thread executé, on ne peut pas le ré-executer (il a été consommé)

In [5]:
t.start()

RuntimeError: threads can only be started once

L'intéret d'avoir un Thread, c'est que pendant les "temps mort" de celui-ci, un autre code peut s'executer.

In [10]:
t = threading.Thread(target=worker)
t.start()
print("ce message est affiché alors que t est en train de travailler")
time.sleep(4)
print("un autre message qui s'affiche pendant que t travaille toujours")
time.sleep(3)
print("nous sommes presque arriver a la fin")

print("nous sommes arriver a la fin")


0
ce message est affiché alors que t est en train de travailler
1
2
3
un autre message qui s'affiche pendant que t travaille toujours
4
5
6
nous sommes presque arriver a la fin
nous sommes arriver a la fin
7


8
9


Tout le code apres `t.join()` ne s'executera qu'apres cette ligne : On attent que le fil rejoinge le fil principal.

In [11]:
t = threading.Thread(target=worker)
t.start()

print("ce message est affiché alors que t est en train de travailler")
time.sleep(4)
print("un autre message qui s'affiche pendant que t travaille toujours")

t.join()   # On attend la fin du thread
print("Ce message arrive seulement une fois que t a fini")

0
ce message est affiché alors que t est en train de travailler
1
2
3
un autre message qui s'affiche pendant que t travaille toujours4

5
6
7
8
9
Ce message arrive seulement une fois que t a fini


## Passer des arguments a un Thread

Si la fonction `target` contient des arguments, on peut les faire passer dans notre `Thread` avec l'entrée `args`

In [ ]:
def worker(n):
    for i in range(n):
        time.sleep(0.5)
        print(i)
    

t = threading.Thread(target=worker, args=(5,))
t.start()

0
1
2
3
4


## Travailler avec plusieurs `Threads`

le threading permet de créer autant de threads qu'on le désire au sein d'un meme programme, permettant ainsi d'utiliser intelligement nos ressources durant certains temps "morts".

Par exemple: Un Thread envoie une requete a un serveur, et pendant qu'il attend une réponse, il se met en "pause" pour laisser la place a un autre thread qui va effectuer un calcul.

In [14]:
import threading
import time

In [15]:
def func1():
    i = 1
    while i < 10:
        print(f"func1 : {i}")
        i += 1
        time.sleep(0.5)

In [16]:
def func2():
    i = 1
    while i < 10:
        print(f"func2 : {i}")
        i *= 2
        time.sleep(0.5)

In [17]:
t1 = threading.Thread(target = func1)
t2 = threading.Thread(target = func2)

t1.start()
t2.start()

func1 : 1
func2 : 1


func1 : 2
func2 : 2
func1 : 3
func2 : 4
func1 : 4
func2 : 8
func1 : 5
func1 : 6
func1 : 7
func1 : 8
func1 : 9


Autre exemple : On peut également créer plusieurs thread a partir d'une meme fonction ! 

In [18]:
def worker():
    for i in range(10):
        time.sleep(0.5)
        print(i)

t1 = threading.Thread(target = worker)
t2 = threading.Thread(target = worker)

In [19]:
t1.start()
t2.start()

0
0
1
1
2
2
3
3
4
4
5
5
6
6
7
7
8
8
9
9


In [20]:
def worker():
    for i in range(10):
        print(i)
        # time.sleep(0.1)

t1 = threading.Thread(target = worker)
t2 = threading.Thread(target = worker)

In [21]:
t1.start()
t2.start()

0
1
2
3
4
5
6
7
8
9
0
1
2
3
4
5
6
7
8
9


Ce qu'on observe: Si un thread n'as pas de temps "mort" (exemple: sleep, temps de chargement, attente réseau), il ne va pas s'arreter pour laisser ton "copain" travailler. le module `threading` nous permet donc de réaliser du "pseudo-parallele". 

## Connaitre le nombre de thread dans notre programe python

la fonction `active_count()` permet de connaitre le nombre de threads actifs dans notre programme.

On observe qu'au sein d'un Jupyter Notebook, il y a plus de `threads` qu'attendu, cela est du au fait que python fait tourner d'autres éléments que notre code dans un Jupyter Notebook : L'interface UI, IPython, etc. On peut obtenier la liste de tous ces threads avec la fonction `threading.enumerate()`

In [23]:
import threading

threading.active_count()

7

In [24]:
for thread in threading.enumerate(): 
    print(thread.name)

MainThread
IOPub
Heartbeat
Thread-1 (_watch_pipe_fd)
Thread-2 (_watch_pipe_fd)
Control
IPythonHistorySavingThread


## Les Threads `Daemon`

Un thread daemon est un thread qui s’arrête automatiquement quand le programme principal s’arrête.

Cela est typiquement utile pour des tâches secondaires (logs, monitoring…).

Attention le Code ci-dessous n'est pas a executer dans un Notebook, car le thread ne se terminera pas automatiquement !
(Le Kernel continuant de tourner au sein du notebook, celui-ci ne met pas fin au daemon).
Vous pouvez en revanche copier coller ce code dans un vrai fichier.py

In [25]:
import threading
import time

def tache_daemon():
    while True:
        print("Thread daemon en cours…")
        time.sleep(1)

def tache_normale():
    for i in range(3):
        print("Thread normal :", i)
        time.sleep(1)

# Création d'un thread daemon
thread_d = threading.Thread(target=tache_daemon, daemon=True)

# Création d'un thread normal
thread_n = threading.Thread(target=tache_normale)

thread_d.start()
thread_n.start()

thread_n.join()   # On attend la fin du thread normal
print("Programme terminé : le thread daemon va s'arrêter automatiquement.")


Thread daemon en cours…
Thread normal : 0
Thread daemon en cours…
Thread normal : 1
Thread daemon en cours…
Thread normal : 2
Thread daemon en cours…
Programme terminé : le thread daemon va s'arrêter automatiquement.


Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en cours…
Thread daemon en

## Verouiller les opérations avec `Lock`

Le probleme avec les opérations concurrentes, c'est qu'elles peuvent parfois donner des résultats indésirables.

Par exemple : Une banque dipose de 100_000 euros. On crée une fonction `retirer` qui vérifie si un montant a retirer ne dépasse pas le solde de la banque.

Mais si 2 opérations sont demandées en meme temps (l'une a 50,000 et l'autre a 60,000) alors la banque pourrait accidentellement se retrouver dans le négatif.

Démonstration:

In [7]:
import time
import threading

In [8]:
bank_account = 100_000

In [9]:
class MontantImpossible(Exception):
    pass

def retirer(montant):
    global bank_account
    print(f"demande en cours : retirer {montant}")
    if montant > bank_account:
        raise MontantImpossible
    else:
        time.sleep(1) # temps nécessaire pour effectuer la transaction
        bank_account -= montant
        print(f"Retrait de {montant}, nouveau solde = {bank_account}")

In [10]:
retirer(80_000)

demande en cours : retirer 80000
Retrait de 80000, nouveau solde = 20000


Créons a présent des `Thread` pour rendre ces opérations concurrentes (et ne pas faire attendre d'autres clients)

In [11]:
bank_account = 100_000

In [ ]:
t1 = threading.Thread(target=retirer, args=(60_000,))
t2 = threading.Thread(target=retirer, args=(50_000,))

t1.start()
t2.start()

demande en cours : retirer 60000
demande en cours : retirer 50000


Retrait de 60000, nouveau solde = 40000
Retrait de 50000, nouveau solde = -10000


In [13]:
print(bank_account)

-10000


La solution : au sein d'un `thread`, on peut verrouiller une partie du code en cours d'execution avec un `lock`, ce qui empeche les autres threads de travailler pendant les temps d'attente de celui-ci.

ce verrouillage est a placer dans un `context manager`

In [14]:
import threading
import time

class MontantImpossible(Exception):
    pass

bank_account = 1000  # exemple
lock = threading.Lock()  # verrou global

def retirer(montant):
    global bank_account
    print(f"demande en cours : retirer {montant}")
    with lock:  # verrouillage automatique à l’entrée, libération à la sortie
        if montant > bank_account:
            raise MontantImpossible
        else:
            time.sleep(1)  # temps nécessaire pour effectuer la transaction
            bank_account -= montant
            print(f"Retrait de {montant}, nouveau solde = {bank_account}")


In [15]:
bank_account = 100_000

t1 = threading.Thread(target=retirer, args=(60_000,))
t2 = threading.Thread(target=retirer, args=(50_000,))

t1.start()
t2.start()

demande en cours : retirer 60000
demande en cours : retirer 50000


Exception in thread Thread-11 (retirer):
Traceback (most recent call last):
  File "/home/guillaume/.pyenv/versions/3.12.9/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/home/guillaume/.pyenv/versions/3.12.9/lib/python3.12/site-packages/ipykernel/ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "/home/guillaume/.pyenv/versions/3.12.9/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_63761/2423950156.py", line 15, in retirer
MontantImpossible


Retrait de 60000, nouveau solde = 40000


## L'utilisation de Sémaphores

Un Semaphore limite le nombre de threads autorisés dans une partie de votre programme. Cela peut etre important pour éviter de saturer la mémoire.

Exemple : limiter à 5 transfert de banque simultanés

In [17]:
sem = threading.Semaphore(3)

def worker(i):
    with sem:
        print(f"Thread {i} commence")
        time.sleep(1)
        print(f"Thread {i} finit")

for i in range(10):
    threading.Thread(target=worker, args=(i,)).start()

Thread 0 commence
Thread 1 commence
Thread 2 commence


Thread 0 finit
Thread 3 commence
Thread 1 finit
Thread 4 commence
Thread 2 finit
Thread 5 commence
Thread 3 finit
Thread 4 finit
Thread 6 commence
Thread 7 commence
Thread 5 finit
Thread 8 commence
Thread 6 finit
Thread 8 finit
Thread 9 commence
Thread 7 finit
Thread 9 finit


# Synchroniser des threads avec `Event` et `Condition`

## Les évenements

Un `Event` sert lorsque un ou plusieurs threads doivent attendre qu’un autre thread déclenche un signal avant de continuer.

Exemple typiques:
- **Synchroniser un démarrage** : Un groupe de threads doit attendre que les données soient prêtes avant de commencer.
- **Stopper proprement un thread** : Un thread effectue une boucle continue et s'arrête lorsque l’événement stop est activé.
- **Orchestration entre threads** : Un thread produit quelque chose → un autre thread attend ce signal avant d’agir.
- **Attendre la fin d’une initialisation** : Un thread initialise un système (ex : connexion réseau), les autres attendent qu’il soit prêt.


## Les conditions

`Event` étant un mécanisme `global`, celui ci déclenche tous les threads qui le suivent.
Ce qui peut donner des soucis de synchronisation (notamment les verrous, comme vu précédemment)


La classe `Condition` permet de résoudre ce probleme en offrance un verrou + une file d’attente de threads + la possibilité de notifier certains threads seulement.

Elle sert pour des synchronisations plus complexes, en particulier lorsqu’un état dépend de variables partagées dont la cohérence doit être protégée par un lock

Par exemple : 

On réalise une architecture `producteur-consommateur`, avec un buffer limité, où :

- Plusieurs producteurs peuvent produire une valeur
- Plusieurs consommateurs peuvent consommer ces valeurs
- Les threads doivent attendre si le buffer est plein (pour les producteurs) ou vide (pour les consommateurs).

_____

Dans l'exemple ci-dessous, Un thread télécharge une configuration, puis plusieurs threads attendent qu’elle soit prête pour travailler

In [18]:
config_ready = threading.Event()

In [19]:
configuration = None

In [20]:
def load_configuration():
    global configuration
    print("[Loader] Chargement de la configuration en cours...")
    time.sleep(3)  # Simule un téléchargement lent
    configuration = {"retry": 5, "timeout": 10}
    print("[Loader] Configuration chargée, signalement...")
    config_ready.set()  # Déclenchement de l'événement

In [21]:
import random

def worker(worker_id):
    print(f"[Worker {worker_id}] En attente de la configuration...")
    
    # Attend le signal du loader
    config_ready.wait()
    
    # Maintenant la config est disponible
    print(f"[Worker {worker_id}] Démarre avec la configuration: {configuration}")
    time.sleep(random.randint(1, 3))
    print(f"[Worker {worker_id}] Travail terminé.")

In [22]:
# Thread qui charge la configuration
threading.Thread(target=load_configuration).start()

# 3 workers qui attendent le signal
for i in range(3):
    threading.Thread(target=worker, args=(i,)).start()

[Loader] Chargement de la configuration en cours...
[Worker 0] En attente de la configuration...
[Worker 1] En attente de la configuration...
[Worker 2] En attente de la configuration...


[Loader] Configuration chargée, signalement...
[Worker 1] Démarre avec la configuration: {'retry': 5, 'timeout': 10}
[Worker 2] Démarre avec la configuration: {'retry': 5, 'timeout': 10}
[Worker 0] Démarre avec la configuration: {'retry': 5, 'timeout': 10}
[Worker 1] Travail terminé.
[Worker 0] Travail terminé.
[Worker 2] Travail terminé.


Dans cet exemple:
1. Les workers démarrent immédiatement, mais attendent l’événement `config_ready.wait()`
2. Le thread loader simule un chargement long.
3. Quand la config est prête, il appelle: `config_ready.set()`
4. Tous les workers débloqués continuent simultanément.

# Transfert de données entre `Threads` a l'aide de `Queues`

Le module `queue` (inclus dans la bibliothèque standard) fournit des structures de données sécurisées pour le multithreading, permetant de partager des données entre plusieurs threads sans risque de conditions de course (race conditions).

https://docs.python.org/3/library/queue.html

En Python, les structures classiques (listes, dicts…) ne sont pas thread-safe.
Le module `queue` propose des files sécurisées grâce à un verrou interne.

On y retrouve les classes suivantes :
- `Queue` : Pour gerer des files FIFO (First In, First Out)
- `LifoQueue`: Pour gerer des files LIFO (Last In, First Out)
- `PriorityQueue`: Pour donner une priorité aux éléments d'une file.
- `SimpleQueue` (depuis Python 3.7) qui est une version plus legere sans `Lock`


In [23]:
from queue import Queue

q = Queue(maxsize=0)  # 0 = taille infinie

| Méthode              | Rôle                                             |
| -------------------- | ------------------------------------------------ |
| `q.put(item)`        | ajoute un élément (bloque si la file est pleine) |
| `q.get()`            | récupère un élément (bloque si la file est vide) |
| `q.put_nowait(item)` | ajoute sans attendre                             |
| `q.get_nowait()`     | récupère sans attendre                           |
| `q.empty()`          | True si la file est vide                         |
| `q.full()`           | True si pleine                                   |
| `q.task_done()`      | indique qu’une tâche est terminée                |
| `q.join()`           | attend que toutes les tâches soient traitées     |


In [25]:
import queue

q = queue.Queue() # FIFO

q.put("A")
q.put("B")

In [26]:
print(q.get())

A


In [27]:
print(q.get())

B


`Queue` permet ainsi de facilement créer des architectures `produceur - consumer`

In [29]:
q = Queue()

def producer():
    for i in range(5):
        print("Produit", i)
        q.put(i)
        time.sleep(1)

def consumer():
    while True:
        item = q.get()
        print("Consomme", item)
        q.task_done()

threading.Thread(target=consumer, daemon=True).start()
threading.Thread(target=producer).start()

Produit 0
Consomme 0


Produit 1
Consomme 1
Produit 2
Consomme 2
Produit 3
Consomme 3
Produit 4
Consomme 4


In [30]:
threading.enumerate()

[<_MainThread(MainThread, started 127969602997120)>,
 <Thread(IOPub, started daemon 127969529984704)>,
 <Heartbeat(Heartbeat, started daemon 127969521592000)>,
 <Thread(Thread-1 (_watch_pipe_fd), started daemon 127969288779456)>,
 <Thread(Thread-2 (_watch_pipe_fd), started daemon 127969280386752)>,
 <ControlThread(Control, started daemon 127969271994048)>,
 <HistorySavingThread(IPythonHistorySavingThread, started 127969262552768)>,
 <Thread(Thread-36 (consumer), started daemon 127969231083200)>]

# En résumé...

- Python ne permet pas de créer des threads 100% paralleles (a cause du GIL)
- Mais il peut "paralleliser" les threads lorsque ces derniers "attendent", ce qui les rends particulierement intéressants pour :
    - Les tâches I/O (réseau, fichiers)
    - Les API: serveurs / clients
    - le Web scrapping
    - Les taches d'arriere plan : Timer, Watcher, Logging, Monitoring, Rafraichir une page etc.. 
    - Les temps d'attente (d'une maniere générale)

Cependant, le `threading` est une technique couteuse, et remplie de piege en Python.

c'est pourquoi, nous verrons par la suite une méthode alternative (plus légere et plus intéressante dans certains cas): Le module `asyncio`

## Rappelez-vous des pieges courrants: 

- Accès concurrent à une variable -> Utilisez `Lock`  
- Threads trop nombreux: Ils consomment mémoire + contexte -> Utilisez `Sémaphores`
- Thread qui vit pour toujours -> Utilisez `daemon`


___

## Exercice : 

Réalisez un programme téléchargeant les fichiers suivants :

```python
urls = [
    "https://images.unsplash.com/photo-1?w=800",
    "https://images.unsplash.com/photo-2?w=800",
    "https://images.unsplash.com/photo-3?w=800",
    "https://images.unsplash.com/photo-4?w=800",
    "https://images.unsplash.com/photo-5?w=800",
    "https://images.unsplash.com/photo-6?w=800",
    "https://images.unsplash.com/photo-7?w=800",
    "https://images.unsplash.com/photo-8?w=800",
    "https://images.unsplash.com/photo-9?w=800",
    "https://images.unsplash.com/photo-10?w=800",
]
```

Dans une version `monothread` et une version `multithread`



In [ ]:
import threading
import requests
import time

# Liste de fichiers à télécharger (images gratuites depuis Unsplash)
urls = [
    "https://images.unsplash.com/photo-1?w=800",
    "https://images.unsplash.com/photo-2?w=800",
    "https://images.unsplash.com/photo-3?w=800",
    "https://images.unsplash.com/photo-4?w=800",
    "https://images.unsplash.com/photo-5?w=800",
    "https://images.unsplash.com/photo-6?w=800",
    "https://images.unsplash.com/photo-7?w=800",
    "https://images.unsplash.com/photo-8?w=800",
    "https://images.unsplash.com/photo-9?w=800",
    "https://images.unsplash.com/photo-10?w=800",
]


# Votre Code ICI

### Correction

In [ ]:
import threading
import requests
import time

# Liste de fichiers à télécharger (images gratuites depuis Unsplash)
urls = [
    "https://images.unsplash.com/photo-1?w=800",
    "https://images.unsplash.com/photo-2?w=800",
    "https://images.unsplash.com/photo-3?w=800",
    "https://images.unsplash.com/photo-4?w=800",
    "https://images.unsplash.com/photo-5?w=800",
    "https://images.unsplash.com/photo-6?w=800",
    "https://images.unsplash.com/photo-7?w=800",
    "https://images.unsplash.com/photo-8?w=800",
    "https://images.unsplash.com/photo-9?w=800",
    "https://images.unsplash.com/photo-10?w=800",
]


In [ ]:
# -------------------------
# Version séquentielle
# -------------------------
def download_file(url):
    r = requests.get(url)
    return len(r.content)

def sequential_download(urls):
    start = time.time()
    for url in urls:
        size = download_file(url)
        print(f"{url} -> {size} bytes")
    end = time.time()
    print(f"Temps total séquentiel : {end - start:.2f} s\n")

In [ ]:
# -------------------------
# Version Threaded
# -------------------------
def threaded_download(urls):
    start = time.time()
    results = {}

    def worker(url):
        size = download_file(url)
        results[url] = size
        print(f"{url} -> {size} bytes")

    threads = []
    for url in urls:
        t = threading.Thread(target=worker, args=(url,))
        t.start()
        threads.append(t)

    for t in threads:
        t.join()

    end = time.time()
    print(f"Temps total threading : {end - start:.2f} s\n")

In [ ]:
# -------------------------
# Lancer la comparaison
# -------------------------
print("Téléchargement séquentiel :")
sequential_download(urls)

print("Téléchargement multi-thread :")
threaded_download(urls)

# [Bonus] Un contre-Exemple de `Threading`

Comme dit dans les vidéos, on peut penser que le `threading` accelere systématiquement les opérations d'entrée / sortie.


**Mais ca n'est pas le cas !**

Pour vous le montrer, j'ai placé au chemin `data/data.csv` un fichier csv de 10,000 lignes.
Je vous invite a tenter d'ouvrir ce fichier, ligne par ligne, et écrire un code avec 2 threads:
- l'une qui lit chaque ligne l'une apres l'autre
- l'autre qui filtre si la ligne est une string ou une valeur numérique.

Vous verrez que l'option `threading` est plus lente ! 